In [ ]:
import os, sys
# ── CAPTURE CELL : Run this FIRST before any other cell ─────────────────────
# All print() output appears in BOTH the cell output AND ps4_clustering_tvm_outputs/console.log
# All plt.show() figures appear in BOTH the cell output AND ps4_clustering_tvm_outputs/<name>.png
OUT_DIR = "ps4_clustering_tvm_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

class _Tee:
    # Writes to Jupyter cell output + log file simultaneously
    def __init__(self, jupyter_stream, log_fh):
        self._jupyter = jupyter_stream
        self._log     = log_fh
        self._is_tee  = True
    def write(self, data):
        try: self._jupyter.write(data); self._jupyter.flush()
        except Exception: pass
        try: self._log.write(data); self._log.flush()
        except Exception: pass
    def flush(self):
        try: self._jupyter.flush()
        except Exception: pass
        try: self._log.flush()
        except Exception: pass
    def __getattr__(self, name): return getattr(self._jupyter, name)

if not getattr(sys.stdout, "_is_tee", False):
    _log_fh    = open(os.path.join(OUT_DIR, "console.log"), "w", buffering=1, encoding="utf-8")
    sys.stdout = _Tee(sys.stdout, _log_fh)
    sys.stderr = _Tee(sys.stderr, _log_fh)
    print("[capture] All output -> cell output + " + os.path.join(OUT_DIR, "console.log"))
else:
    print("[capture] Already active.")

try:
    import matplotlib.pyplot as _plt
    if not getattr(_plt.show, "_wrapped", False):
        _orig_show = _plt.show
        def _show(*a, _cnt=[0], **k):
            for _fig_num in _plt.get_fignums():
                _fig   = _plt.figure(_fig_num)
                _title = (
                    (_fig._suptitle.get_text().strip() if _fig._suptitle else None)
                    or (_fig.axes[0].get_title().strip() if _fig.axes else None)
                    or f"fig_{_cnt[0]:02d}"
                )
                _safe  = "".join(
                    c if c.isalnum() or c in "-_ " else "_" for c in _title
                ).strip().replace(" ", "_")[:60]
                _fname = os.path.join(OUT_DIR, f"{_safe}_{_cnt[0]:02d}.png")
                try:
                    _fig.savefig(_fname, dpi=110, bbox_inches="tight")
                    print(f"[capture] fig saved -> {_fname}")
                    _cnt[0] += 1
                except Exception as _e:
                    print(f"[capture] fig save failed: {_e}")
            return _orig_show(*a, **k)
        _show._wrapped = True
        _plt.show = _show
    print(f"[capture] matplotlib patched -> figures saved to {OUT_DIR}/")
except Exception as _e:
    print("[capture] matplotlib not ready:", _e)


# PS4 — Fault Clustering : TVM

In [ ]:
# ── CELL 1 : Install ────────────────────────────────────────────────────────
import subprocess, sys
pkgs = ["kneed","scikit-learn","matplotlib","seaborn","pandas","numpy","pyarrow","boto3","s3fs","mlflow"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Install complete.")

In [ ]:
# ── CELL 2 : Imports ────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble      import IsolationForest
from sklearn.cluster       import KMeans, DBSCAN
from sklearn.pipeline      import Pipeline as skPipeline
from sklearn.mixture       import GaussianMixture
from sklearn.metrics       import silhouette_score
from sklearn.neighbors     import NearestNeighbors
from kneed                 import KneeLocator
import mlflow
import mlflow.sklearn

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_style("whitegrid")
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ──────────────────────────────────────────────────
S3_BUCKET   = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX = "chicago/gold/device_ps1_daily"
REGION      = "us-east-1"
GOLD_S3     = f"s3://{S3_BUCKET}/{GOLD_PREFIX}"

DEVICE_TYPES  = ["TVM", "GATE", "VALIDATOR"]
DEV_COL       = "mars_device_category"
DEVICE_ID_COL = "DEVICE_ID"
DATE_COL      = "event_date"
TARGET_COL    = "will_fail_3d"

# ── Sampling caps (None = use full device data) ────────────────────────────
# Daily data: TVM ~360k  GATE ~670k  VALIDATOR ~2M rows
# KMeans / PCA / IF / DBSCAN / GMM  — all scale well enough for full daily data
S_KM      = None      # full data
S_PCA     = None      # full data
S_DBSCAN  = None      # full data  (O(n log n) with ball tree)
S_GMM     = None      # full data  (warn: VALIDATOR EM may take 30-60 min)
RANDOM_STATE     = 42

# ── KMeans / GMM search range ─────────────────────────────────────────────
K_MIN, K_MAX     = 2, 10
IF_CONTAMINATION = 0.05

# ── DBSCAN ────────────────────────────────────────────────────────────────
DBSCAN_K_NEIGHBORS  = 5
DBSCAN_MIN_SAMPLES  = 5

# ── Feature exclusions (PS1 daily columns) ────────────────────────────────
EXCLUDE_COLS = {
    DEV_COL, DEVICE_ID_COL, "DEVICE_NAME", DATE_COL, TARGET_COL,
    "mars_device_category", "device_id", "device_name",
    "split", "event_date", "FACILITY_ID", "FACILITY_NAME",
    "STATION_ID", "STATION_NAME", "OPERATOR_ID", "BUS_ID",
    "DEVICE_KEY", "TRANSIT_ARRAY_ID", "ARRAY_POSITION",
    "will_fail_3d", "will_fail_7d", "max_failure_level",
    "oos_flag", "oos_event_count",
}
print(f"Config ready.  OUT_DIR={OUT_DIR}")
print(f"Gold table  : {GOLD_S3}  (DAILY)")
print(f"Sampling: KM=full  PCA=full  DBSCAN=full  GMM=full")

# ── MLflow ────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
MLFLOW_EXPERIMENT   = "chicago-ps4-fault-clustering"
MLFLOW_MODEL_NAME   = "chicago-ps4-tvm-fault-clustering"
MODEL_REGISTRY_NAME = "chicago-ps4-tvm-fault-clustering"
COMMON_TAGS = {"ps": "PS4", "city": "chicago", "target": "fault_clustering", "model_version": "v1"}
PIPELINE_RESULTS = {}
# ── S3 export (optional CELL 15 — same pattern as PS4 anomaly CELL 19) ───────
ENABLE_S3_EXPORT = True
PS4_CLUSTER_S3_PREFIX = "chicago/ps4/clustering"
CITY_CODE = "CHI"


In [ ]:
# ── CELL 3b : MLflow initialisation ─────────────────────────────────────────
import mlflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
print(f"MLflow experiment : {MLFLOW_EXPERIMENT}")
print(f"MLflow tracking   : {MLFLOW_TRACKING_URI}")
print(f"Model registry    : {MLFLOW_MODEL_NAME}")

def _track_pipeline(name, sil, run_id, flavor="sklearn"):
    PIPELINE_RESULTS[name] = {"sil": sil, "run_id": run_id, "flavor": flavor}

In [ ]:
# ── CELL 4 : Load PS4 Gold table (Parquet on S3) ───────────────────────────
print(f"Reading: {GOLD_S3}")
df_full = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
print(f"Loaded : {df_full.shape[0]:,} rows x {df_full.shape[1]} cols")
print(f"\nColumns ({len(df_full.columns)}):")
print(df_full.columns.tolist())

# ── Auto-detect DATE_COL ──────────────────────────────────────────────────────
if DATE_COL not in df_full.columns:
    _cands = [c for c in df_full.columns if any(k in c.lower() for k in ["hour","date","time","dt"])]
    print(f"\n[WARN] DATE_COL '{DATE_COL}' not found.  Candidates: {_cands}")
    DATE_COL = _cands[0] if _cands else None
    if DATE_COL: print(f"  Using '{DATE_COL}' as DATE_COL")
if DATE_COL:
    df_full[DATE_COL] = pd.to_datetime(df_full[DATE_COL], errors="coerce")

# ── Auto-detect DEV_COL ───────────────────────────────────────────────────────
if DEV_COL not in df_full.columns:
    _cands = [c for c in df_full.columns if any(k in c.lower() for k in ["device","dev","category","type"])]
    print(f"\n[WARN] DEV_COL '{DEV_COL}' not found.  Candidates: {_cands}")
    DEV_COL = _cands[0] if _cands else None
    if DEV_COL: print(f"  Using '{DEV_COL}' as DEV_COL")

if DEV_COL and DEV_COL in df_full.columns:
    df_full[DEV_COL] = df_full[DEV_COL].astype(str).str.strip().str.upper()
    print(f"\nDevice-type counts ({DEV_COL}):")
    print(df_full[DEV_COL].value_counts().to_string())
else:
    print(f"\n[WARN] No device column found — check column list above.")

# ── Auto-detect DEVICE_ID_COL ────────────────────────────────────────────────
if DEVICE_ID_COL not in df_full.columns:
    _cands = [c for c in df_full.columns if "device" in c.lower() and "id" in c.lower()]
    print(f"\n[WARN] DEVICE_ID_COL '{DEVICE_ID_COL}' not found.  Candidates: {_cands}")
    DEVICE_ID_COL = _cands[0] if _cands else None
    if DEVICE_ID_COL: print(f"  Using '{DEVICE_ID_COL}' as DEVICE_ID_COL")


In [ ]:
# ── CELL 5 : Feature preparation ────────────────────────────────────────────
_excl = {c.lower() for c in EXCLUDE_COLS}
FEATURE_COLS = [
    c for c in df_full.columns
    if c.lower() not in _excl
    and pd.api.types.is_numeric_dtype(df_full[c])
]
df_full[FEATURE_COLS] = df_full[FEATURE_COLS].fillna(0)
print(f"Feature columns : {len(FEATURE_COLS)}")
print(FEATURE_COLS)


In [ ]:
# ── CELL 6 : Helper functions ───────────────────────────────────────────────

def _samp(dev, n):
    d = df_full[df_full[DEV_COL] == dev].copy()
    print(f"  {dev}: {len(d):,} daily rows", end="")
    # Aggregate to device grain: mean across all days → 1 row per DEVICE_KEY
    _num = [c for c in FEATURE_COLS + ([TARGET_COL] if TARGET_COL in d.columns else [])
            if c in d.columns]
    d = d.groupby("DEVICE_KEY")[_num].mean().reset_index()
    if DEVICE_ID_COL in df_full.columns:
        _id = (df_full[df_full[DEV_COL] == dev][["DEVICE_KEY", DEVICE_ID_COL]]
               .drop_duplicates("DEVICE_KEY"))
        d = d.merge(_id, on="DEVICE_KEY", how="left")
    print(f"  -> {len(d):,} unique devices")
    if n is not None and len(d) > n:
        d = d.sample(n, random_state=RANDOM_STATE)
        print(f"  -> sampled to {n:,} devices")
    print()
    return d

def _scale(d):
    feat = [c for c in FEATURE_COLS if c in d.columns]
    return StandardScaler().fit_transform(d[feat].values), feat

def _pca95(Xs):
    full = PCA(random_state=RANDOM_STATE).fit(Xs)
    cum  = np.cumsum(full.explained_variance_ratio_)
    n    = int(np.argmax(cum >= 0.95)) + 1
    Xp   = PCA(n_components=n, random_state=RANDOM_STATE).fit_transform(Xs)
    return Xp, n, cum

def _pca2(Xs):
    return PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(Xs)

def _elbow(Xdata, lbl=""):
    kr   = range(K_MIN, K_MAX + 1)
    wcss = [KMeans(n_clusters=k, random_state=RANDOM_STATE,
                   n_init=10).fit(Xdata).inertia_ for k in kr]
    kl   = KneeLocator(list(kr), wcss, curve="convex", direction="decreasing")
    opt  = kl.knee if kl.knee else K_MIN + 1
    print(f"  {lbl} optimal K = {opt}")
    return opt, list(kr), wcss

def _ax_elbow(ax, kr, wcss, opt, title):
    ax.plot(kr, wcss, "bo-", ms=5)
    ax.axvline(opt, color="red", ls="--", lw=1.5, label=f"K={opt}")
    ax.set(xlabel="K", ylabel="WCSS", title=title); ax.legend()

def _ax_var(ax, cum, n):
    ax.plot(range(1, len(cum)+1), cum*100, "b-", lw=1.5)
    ax.axhline(95, color="red", ls="--", lw=1, label="95%")
    ax.axvline(n, color="orange", ls="--", lw=1, label=f"n={n}")
    ax.set(xlabel="Components", ylabel="Cumul. Variance %", title="PCA Variance Explained")
    ax.legend()

def _ax_if_scatter(ax, X2, lbl_if, title, x="Dim1", y="Dim2"):
    from matplotlib.patches import Patch as _P
    col = np.where(lbl_if == 1, "#1565C0", "#E53935")
    ax.scatter(X2[:, 0], X2[:, 1], c=col, alpha=0.3, s=6)
    ax.legend(handles=[_P(color="#1565C0", label="Normal"),
                       _P(color="#E53935", label="Anomaly")])
    ax.set(title=title, xlabel=x, ylabel=y)

def _ax_cl(ax, X2, cl, nc, title, x="Dim1", y="Dim2"):
    pal = sns.color_palette("tab10", nc)
    for c in range(nc):
        m = cl == c
        ax.scatter(X2[m, 0], X2[m, 1], c=[pal[c]], alpha=0.4, s=6, label=f"C{c}")
    ax.set(title=title, xlabel=x, ylabel=y); ax.legend(markerscale=3)

def _ax_bar(ax, cl, nc, title):
    pal = sns.color_palette("tab10", nc)
    ax.bar(range(nc), [(cl == k).sum() for k in range(nc)], color=pal)
    ax.set(title=title, xlabel="Cluster", ylabel="Count")

def _ax_anom(ax, df_sub, cl_col, nc, title):
    if TARGET_COL in df_sub.columns:
        pal = sns.color_palette("tab10", nc)
        ar  = df_sub.groupby(cl_col)[TARGET_COL].mean()
        ax.bar(ar.index, ar.values, color=[pal[i % nc] for i in ar.index])
        ax.set(title=title, xlabel="Cluster", ylabel="Failure Rate")
    else:
        ax.text(0.5, 0.5, "No target col", ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(title)

def _profile(df_sub, cl_col, feat, lbl, top_n=15):
    print(f"\n-- {lbl} profile (mean, top {top_n}) --")
    _avail = [c for c in feat if c in df_sub.columns]
    if _avail:
        _gm = df_sub[_avail].mean()
        _cm = df_sub.groupby(cl_col)[_avail].mean()
        _disc = (_cm - _gm).abs().max(axis=0).sort_values(ascending=False).head(top_n).index.tolist()
        print(df_sub.groupby(cl_col)[_disc].mean().round(3).T.to_string())
    if DEVICE_ID_COL in df_sub.columns:
        for cl in sorted(df_sub[cl_col].unique()):
            m = df_sub[cl_col] == cl
            print(f"  Cluster {cl}: {m.sum():,} rows | "
                  f"{df_sub.loc[m, DEVICE_ID_COL].nunique()} devices")

def _save(fig, fname):
    p = f"{OUT_DIR}/{fname}"
    fig.savefig(p, dpi=100, bbox_inches="tight")
    plt.show(); plt.close(fig)
    print(f"  Saved: {p}")

print("Helpers ready.")

# ── EXTENDED HELPERS (v5) ────────────────────────────────────────────────────

def _cluster_summary(df_sub, cl_col, feat):
    # Returns DataFrame: Cluster | Size | Pct% | FailRate | DeviceCount | TopFeatures
    rows = []
    total = max(len(df_sub), 1)
    all_cl = sorted(df_sub[cl_col].unique())
    _avail_feat = [c for c in feat if c in df_sub.columns]
    # mean of each feature per cluster -> find top discriminating features
    _means = df_sub.groupby(cl_col)[_avail_feat].mean() if _avail_feat else None
    for cl in all_cl:
        mask = df_sub[cl_col] == cl
        size = int(mask.sum())
        fail = float(df_sub.loc[mask, TARGET_COL].mean()) if TARGET_COL in df_sub.columns and size else float("nan")
        dcnt = int(df_sub.loc[mask, DEVICE_ID_COL].nunique()) if DEVICE_ID_COL in df_sub.columns else -1
        # top 3 features by abs deviation from global mean
        if _means is not None and cl in _means.index and len(_avail_feat) >= 3:
            _gm = df_sub[_avail_feat].mean()
            _dev = (_means.loc[cl] - _gm).abs().nlargest(3)
            top3 = ", ".join(_dev.index.tolist())
        else:
            top3 = ""
        label = "Noise" if cl == -1 else str(cl)
        rows.append({
            "Cluster": label, "Size": size,
            "Pct%": round(100 * size / total, 1),
            "FailRate": f"{fail:.4f}" if not (fail != fail) else "N/A",
            "Devices": dcnt,
            "TopFeatures": top3,
        })
    return pd.DataFrame(rows)


def _device_clusters(df_sub, cl_col):
    # Returns dict {cluster_label: sorted list of device IDs}
    result = {}
    if DEVICE_ID_COL not in df_sub.columns:
        return result
    for cl in sorted(df_sub[cl_col].unique()):
        mask = df_sub[cl_col] == cl
        ids = sorted(df_sub.loc[mask, DEVICE_ID_COL].dropna().astype(str).unique().tolist())
        label = "Noise" if cl == -1 else cl
        result[label] = ids
    return result


def _print_device_list(df_sub, cl_col, pipe_name):
    # Print device IDs per cluster and save CSV
    dev_map = _device_clusters(df_sub, cl_col)
    if not dev_map:
        print("  (DEVICE_ID_COL not found — device listing skipped)")
        return
    print(f"\n{'='*60}")
    print(f"  DEVICE IDs PER CLUSTER — {pipe_name}")
    print(f"{'='*60}")
    rows = []
    for cl, ids in dev_map.items():
        preview = ", ".join(str(x) for x in ids[:20])
        suffix = f" ... +{len(ids)-20} more" if len(ids) > 20 else ""
        print(f"  Cluster {str(cl):6s} | {len(ids):4d} devices | {preview}{suffix}")
        for did in ids:
            rows.append({"cluster": cl, "device_id": did})


def _ax_heatmap(ax, df_sub, cl_col, feat, title, top_n=12):
    # Heatmap of normalized cluster means for top_n features
    _avail = [c for c in feat if c in df_sub.columns][:top_n]
    if not _avail:
        ax.text(0.5, 0.5, "No features", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title); return
    _means = df_sub.groupby(cl_col)[_avail].mean()
    _rng = (_means.max() - _means.min()).replace(0, 1)
    _norm = (_means - _means.min()) / _rng
    _norm.columns = [c[:14] for c in _norm.columns]
    _xlbls = [f"C{c}" if c != -1 else "Noise" for c in _norm.index]
    try:
        sns.heatmap(_norm.T, ax=ax, cmap="YlOrRd", linewidths=0.3,
                    cbar_kws={"shrink": 0.6},
                    xticklabels=_xlbls, yticklabels=_norm.columns.tolist())
    except Exception:
        ax.text(0.5, 0.5, "Heatmap error", ha="center", va="center", transform=ax.transAxes)
    ax.set(title=title); ax.tick_params(axis="y", labelsize=7); ax.tick_params(axis="x", labelsize=8)


def _ax_devcount(ax, df_sub, cl_col, title):
    # Bar chart: unique device count per cluster
    if DEVICE_ID_COL not in df_sub.columns:
        ax.text(0.5, 0.5, "No device col", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title); return
    dcnt = df_sub.groupby(cl_col)[DEVICE_ID_COL].nunique()
    nc = len(dcnt)
    pal = sns.color_palette("tab10", max(nc, 1))
    clabels = [f"C{c}" if c != -1 else "Noise" for c in dcnt.index]
    ax.bar(clabels, dcnt.values, color=[pal[i % 10] for i in range(nc)])
    for i, v in enumerate(dcnt.values):
        ax.text(i, v + 0.4, str(v), ha="center", va="bottom", fontsize=8)
    ax.set(title=title, xlabel="Cluster", ylabel="Unique Devices")


def _ax_timeline(ax, df_sub, cl_col, title):
    # Line chart: monthly record count per cluster
    if DATE_COL and DATE_COL in df_sub.columns:
        try:
            _tmp = df_sub[[cl_col, DATE_COL]].copy()
            _tmp["_ym"] = pd.to_datetime(_tmp[DATE_COL], errors="coerce").dt.to_period("M").astype(str)
            _ct = _tmp.groupby(["_ym", cl_col]).size().unstack(fill_value=0)
            nc = _ct.shape[1]
            pal = sns.color_palette("tab10", max(nc, 1))
            for i, c in enumerate(_ct.columns):
                lbl = f"C{c}" if c != -1 else "Noise"
                ax.plot(_ct.index, _ct[c], label=lbl, color=pal[i % 10], lw=1.5)
            ax.set(title=title, xlabel="Month", ylabel="Records")
            ax.tick_params(axis="x", rotation=45, labelsize=6)
            ax.legend(fontsize=7, ncol=2)
        except Exception as _e:
            ax.text(0.5, 0.5, f"Timeline err:\n{_e}", ha="center", va="center",
                    transform=ax.transAxes, fontsize=7)
            ax.set_title(title)
    else:
        ax.text(0.5, 0.5, "No date col", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title)


def _ax_summary_text(ax, df_sum, title):
    # Render summary DataFrame as monospace text block in axes
    ax.axis("off")
    ax.set_title(title, fontsize=9, fontweight="bold")
    txt = df_sum.to_string(index=False)
    ax.text(0.02, 0.97, txt, transform=ax.transAxes, va="top", ha="left",
            fontsize=7.5, fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="#f0f4f8", alpha=0.9))


print("Extended helpers loaded.")


## TVM — Fault Clustering
*(Plain KMeans → PCA+KM → PCA+IF+KM → DBSCAN → GMM)  — 5 pipelines, full device-grain data, no sampling*

In [ ]:
# ── KMeans ──────────────────────────────────────────────

In [ ]:
# ── CELL 7 : [TVM] Pipeline 1 — K-Means (plain) ──────────────────────
import time as _time; _t0 = _time.time()
_DEV = "TVM"
print("\n" + "="*70)
print(f"  {_DEV}  |  Pipeline 1: K-Means (plain scaled features)")
print("  Algorithm : Standard K-Means on all standardized features (no dim reduction).")
print("  Optimal K : Elbow of WCSS curve via KneeLocator.")
print("  Purpose   : Baseline grouping — reveals natural clusters in raw feature space.")
print("="*70)

_d = _samp(_DEV, S_KM)
_Xs, _feat = _scale(_d)
print(f"\n  Rows      : {len(_d):,}   |   Features: {len(_feat)}")

_opt, _kr, _wcss = _elbow(_Xs, f"{_DEV} KMeans")
_km = KMeans(n_clusters=_opt, random_state=RANDOM_STATE, n_init=10)
_d  = _d.copy(); _d["cluster"] = _km.fit_predict(_Xs)
_sil = silhouette_score(_Xs, _d["cluster"])
print(f"  Optimal K : {_opt}   |   Silhouette: {_sil:.4f}")
_X2 = _pca2(_Xs)

# ── Summary table & device list ───────────────────────────────────────────────
_sum = _cluster_summary(_d, "cluster", _feat)
print(f"\n{'─'*60}\n  CLUSTER SUMMARY — {_DEV} KMeans\n{'─'*60}")
print(_sum.to_string(index=False))
_print_device_list(_d, "cluster", f"{_DEV} KMeans")

# ── 3x3 plot grid ─────────────────────────────────────────────────────────────
_fig = plt.figure(figsize=(22, 15))
_fig.suptitle(f"{_DEV} — Pipeline 1: K-Means (plain)  |  K={_opt}  sil={_sil:.3f}",
              fontsize=14, fontweight="bold")
_gs = gridspec.GridSpec(3, 3, figure=_fig, hspace=0.50, wspace=0.38)
_ax_elbow(_fig.add_subplot(_gs[0, 0]), _kr, _wcss, _opt, f"WCSS Elbow  K_opt={_opt}")
_ax_cl(_fig.add_subplot(_gs[0, 1]), _X2, _d["cluster"].values, _opt,
       f"Clusters 2D PCA view  sil={_sil:.3f}", "PC1", "PC2")
_ax_bar(_fig.add_subplot(_gs[0, 2]), _d["cluster"].values, _opt, "Records per Cluster")
_ax_anom(_fig.add_subplot(_gs[1, 0]), _d, "cluster", _opt, "Failure Rate per Cluster")
_ax_heatmap(_fig.add_subplot(_gs[1, 1]), _d, "cluster", _feat, "Feature Means Heatmap (top 12)")
_ax_devcount(_fig.add_subplot(_gs[1, 2]), _d, "cluster", "Unique Devices per Cluster")
_ax_timeline(_fig.add_subplot(_gs[2, 0]), _d, "cluster", "Monthly Records per Cluster")
_ax_summary_text(_fig.add_subplot(_gs[2, 1]), _sum, f"{_DEV} K-Means Cluster Summary")
_fig.add_subplot(_gs[2, 2]).axis("off")
_fig.tight_layout()
_save(_fig, f"{_DEV.lower()}_1_kmeans.png")
with mlflow.start_run(run_name=f"{_DEV}_P1_KMeans", tags={**COMMON_TAGS, "device_type": _DEV, "pipeline": "KMeans"}) as run:
    mlflow.log_params({"algorithm": "KMeans", "n_clusters": _opt, "k_min": K_MIN, "k_max": K_MAX, "random_state": RANDOM_STATE})
    mlflow.log_metrics({"silhouette_score": round(_sil, 4), "inertia": round(float(_km.inertia_), 2), "n_devices": len(_d)})
    _cdist = {f"cluster_{k}_size": int((_d['cluster']==k).sum()) for k in range(_opt)}
    mlflow.log_metrics(_cdist)
    mlflow.log_artifact(f"{OUT_DIR}/{_DEV.lower()}_1_kmeans.png")
    _sk_scaler = StandardScaler().fit(_d[_feat].values)
    _sk_pipe = skPipeline([("scaler", _sk_scaler), ("kmeans", KMeans(n_clusters=_opt, random_state=RANDOM_STATE, n_init=10).fit(_sk_scaler.transform(_d[_feat].values)))])
    mlflow.sklearn.log_model(_sk_pipe, "model")
    _track_pipeline("P1_KMeans", _sil, run.info.run_id)
_profile(_d, "cluster", _feat, f"{_DEV} KMeans")
print(f"\n  [CELL 14 : [{_DEV}] Pipeline 1 K-Means]  Elapsed: {(_time.time()-_t0)/60:.2f} min")


In [ ]:
# ── with PCA ────────────────────────────────────────────

In [ ]:
# ── CELL 8 : [TVM] Pipeline 2 — PCA + K-Means ───────────────────────
import time as _time; _t0 = _time.time()
_DEV = "TVM"
print("\n" + "="*70)
print(f"  {_DEV}  |  Pipeline 2: PCA + K-Means")
print("  Algorithm : PCA retaining 95% variance -> K-Means on principal components.")
print("  Benefit   : Reduces noise and collinearity before clustering.")
print("  Optimal K : Elbow of WCSS on PCA-reduced space.")
print("="*70)

_d = _samp(_DEV, S_PCA)
_Xs, _feat = _scale(_d)
_Xp, _npc, _cum = _pca95(_Xs)
print(f"\n  Rows      : {len(_d):,}   |   Features: {len(_feat)}")
print(f"  PCA comps : {_npc}  (95% variance retained)")

_opt, _kr, _wcss = _elbow(_Xp, f"{_DEV} PCA+KM")
_km = KMeans(n_clusters=_opt, random_state=RANDOM_STATE, n_init=10)
_d  = _d.copy(); _d["cluster"] = _km.fit_predict(_Xp)
_sil = silhouette_score(_Xp, _d["cluster"])
print(f"  Optimal K : {_opt}   |   Silhouette: {_sil:.4f}")

# ── Summary table & device list ───────────────────────────────────────────────
_sum = _cluster_summary(_d, "cluster", _feat)
print(f"\n{'─'*60}\n  CLUSTER SUMMARY — {_DEV} PCA+KMeans\n{'─'*60}")
print(_sum.to_string(index=False))
_print_device_list(_d, "cluster", f"{_DEV} PCA+KMeans")

# ── 3x3 plot grid ─────────────────────────────────────────────────────────────
_fig = plt.figure(figsize=(22, 15))
_fig.suptitle(
    f"{_DEV} — Pipeline 2: PCA + K-Means  |  n_pca={_npc}  K={_opt}  sil={_sil:.3f}",
    fontsize=14, fontweight="bold")
_gs = gridspec.GridSpec(3, 3, figure=_fig, hspace=0.50, wspace=0.38)
_ax_var(_fig.add_subplot(_gs[0, 0]), _cum, _npc)
_ax_elbow(_fig.add_subplot(_gs[0, 1]), _kr, _wcss, _opt, f"WCSS Elbow  K_opt={_opt}")
_ax_cl(_fig.add_subplot(_gs[0, 2]), _Xp[:, :2], _d["cluster"].values, _opt,
       f"Clusters PC1-PC2  sil={_sil:.3f}", "PC1", "PC2")
_ax_bar(_fig.add_subplot(_gs[1, 0]), _d["cluster"].values, _opt, "Records per Cluster")
_ax_anom(_fig.add_subplot(_gs[1, 1]), _d, "cluster", _opt, "Failure Rate per Cluster")
_ax_heatmap(_fig.add_subplot(_gs[1, 2]), _d, "cluster", _feat, "Feature Means Heatmap (top 12)")
_ax_devcount(_fig.add_subplot(_gs[2, 0]), _d, "cluster", "Unique Devices per Cluster")
_ax_timeline(_fig.add_subplot(_gs[2, 1]), _d, "cluster", "Monthly Records per Cluster")
_ax_summary_text(_fig.add_subplot(_gs[2, 2]), _sum, f"{_DEV} PCA+KMeans Summary")
_fig.tight_layout()
_save(_fig, f"{_DEV.lower()}_2_pca_kmeans.png")
with mlflow.start_run(run_name=f"{_DEV}_P2_PCA_KMeans", tags={**COMMON_TAGS, "device_type": _DEV, "pipeline": "PCA_KMeans"}) as run:
    mlflow.log_params({"algorithm": "PCA_KMeans", "n_clusters": _opt, "pca_components": _npc, "variance_threshold": 0.95, "k_min": K_MIN, "k_max": K_MAX})
    mlflow.log_metrics({"silhouette_score": round(_sil, 4), "inertia": round(float(_km.inertia_), 2), "n_devices": len(_d), "pca_n_components": _npc})
    _cdist = {f"cluster_{k}_size": int((_d['cluster']==k).sum()) for k in range(_opt)}
    mlflow.log_metrics(_cdist)
    mlflow.log_artifact(f"{OUT_DIR}/{_DEV.lower()}_2_pca_kmeans.png")
    _sk_scaler = StandardScaler().fit(_d[_feat].values)
    _Xs_fit = _sk_scaler.transform(_d[_feat].values)
    _sk_pca = PCA(n_components=_npc, random_state=RANDOM_STATE).fit(_Xs_fit)
    _sk_pipe = skPipeline([("scaler", _sk_scaler), ("pca", _sk_pca), ("kmeans", KMeans(n_clusters=_opt, random_state=RANDOM_STATE, n_init=10).fit(_sk_pca.transform(_Xs_fit)))])
    mlflow.sklearn.log_model(_sk_pipe, "model")
    _track_pipeline("P2_PCA_KMeans", _sil, run.info.run_id)
_profile(_d, "cluster", _feat, f"{_DEV} PCA+KMeans")
print(f"\n  [CELL 15 : [{_DEV}] Pipeline 2 PCA+KMeans]  Elapsed: {(_time.time()-_t0)/60:.2f} min")


In [ ]:
# ── PCA with Isolation Forest ───────────────────────────

In [ ]:
# ── CELL 11 : [TVM] Pipeline 5 — PCA + Isolation Forest + K-Means ────
import time as _time; _t0 = _time.time()
_DEV = "TVM"
print("\n" + "="*70)
print(f"  {_DEV}  |  Pipeline 5: PCA -> Isolation Forest -> K-Means")
print("  Step 1: PCA (95% variance) reduces feature space.")
print("  Step 2: Isolation Forest removes outlier rows.")
print("  Step 3: K-Means clusters the clean inlier set in PCA space.")
print("  Purpose: Cleaner clusters — removes noisy/rare anomalies first.")
print("="*70)

_d = _samp(_DEV, S_PCA)
_Xs, _feat = _scale(_d)
_Xp, _npc, _cum = _pca95(_Xs)
print(f"\n  Rows: {len(_d):,}   |   Features: {len(_feat)}   |   PCA components: {_npc}")

_if   = IsolationForest(contamination=IF_CONTAMINATION, random_state=RANDOM_STATE, n_jobs=-1)
_if_lbl   = _if.fit_predict(_Xp)
_if_score = _if.score_samples(_Xp)
_n_anom  = (_if_lbl == -1).sum()
print(f"  IF anomalies removed: {_n_anom:,}  ({_n_anom/len(_d):.1%})")
_d = _d.copy(); _d["if_label"] = _if_lbl; _d["if_score"] = _if_score

_Xp_clean = _Xp[_if_lbl == 1]
_d_clean  = _d[_if_lbl == 1].copy()

_opt, _kr, _wcss = _elbow(_Xp_clean, f"{_DEV} PCA+IF+KM")
_km  = KMeans(n_clusters=_opt, random_state=RANDOM_STATE, n_init=10)
_d_clean = _d_clean.copy(); _d_clean["cluster"] = _km.fit_predict(_Xp_clean)
_sil = silhouette_score(_Xp_clean, _d_clean["cluster"]) if _opt > 1 else float("nan")
print(f"  Optimal K : {_opt}   |   Silhouette: {_sil:.4f}   (on {len(_d_clean):,} inliers)")

# ── Summary table & device list ───────────────────────────────────────────────
_sum = _cluster_summary(_d_clean, "cluster", _feat)
print(f"\n{'─'*60}\n  CLUSTER SUMMARY — {_DEV} PCA->IF->KMeans (inliers only)\n{'─'*60}")
print(_sum.to_string(index=False))
_print_device_list(_d_clean, "cluster", f"{_DEV} PCA+IF+KMeans")

# ── 3x3 plot grid ─────────────────────────────────────────────────────────────
_fig = plt.figure(figsize=(22, 15))
_fig.suptitle(
    f"{_DEV} — Pipeline 5: PCA -> IF -> K-Means  |  n_pca={_npc}  K={_opt}  sil={_sil:.3f}",
    fontsize=14, fontweight="bold")
_gs = gridspec.GridSpec(3, 3, figure=_fig, hspace=0.50, wspace=0.38)
_ax_var(_fig.add_subplot(_gs[0, 0]), _cum, _npc)

# IF score histogram
_ax1 = _fig.add_subplot(_gs[0, 1])
_ax1.hist(_if_score[_if_lbl ==  1], bins=40, color="#1565C0", alpha=0.6, label="Inlier")
_ax1.hist(_if_score[_if_lbl == -1], bins=40, color="#E53935", alpha=0.6, label="Outlier")
_ax1.set(title="IF Anomaly Score Distribution", xlabel="IF Score"); _ax1.legend(fontsize=8)

_ax_cl(_fig.add_subplot(_gs[0, 2]), _Xp_clean[:, :2], _d_clean["cluster"].values, _opt,
       f"Inlier Clusters PC1-PC2  sil={_sil:.3f}", "PC1", "PC2")
_ax_elbow(_fig.add_subplot(_gs[1, 0]), _kr, _wcss, _opt, f"WCSS Elbow (inliers)  K={_opt}")
_ax_anom(_fig.add_subplot(_gs[1, 1]), _d_clean, "cluster", _opt, "Failure Rate per Cluster")
_ax_heatmap(_fig.add_subplot(_gs[1, 2]), _d_clean, "cluster", _feat, "Feature Means Heatmap")
_ax_bar(_fig.add_subplot(_gs[2, 0]), _d_clean["cluster"].values, _opt, "Records per Cluster (inliers)")
_ax_devcount(_fig.add_subplot(_gs[2, 1]), _d_clean, "cluster", "Unique Devices per Cluster")
_ax_summary_text(_fig.add_subplot(_gs[2, 2]), _sum, f"{_DEV} PCA+IF+KMeans Summary")
_fig.tight_layout()
_save(_fig, f"{_DEV.lower()}_5_pca_if_kmeans.png")
with mlflow.start_run(run_name=f"{_DEV}_P5_PCA_IF_KMeans", tags={**COMMON_TAGS, "device_type": _DEV, "pipeline": "PCA_IF_KMeans"}) as run:
    mlflow.log_params({"algorithm": "PCA_IF_KMeans", "n_clusters": _opt, "pca_components": _npc, "if_contamination": IF_CONTAMINATION, "k_min": K_MIN, "k_max": K_MAX})
    mlflow.log_metrics({"silhouette_score": round(float(_sil) if _sil==_sil else 0.0, 4), "inertia": round(float(_km.inertia_), 2), "n_devices": len(_d), "n_inliers": len(_d_clean), "n_anomalies": int(_n_anom)})
    _cdist = {f"cluster_{k}_size": int((_d_clean['cluster']==k).sum()) for k in range(_opt)}
    mlflow.log_metrics(_cdist)
    mlflow.log_artifact(f"{OUT_DIR}/{_DEV.lower()}_5_pca_if_kmeans.png")
    mlflow.sklearn.log_model(_km, "model")
    mlflow.sklearn.log_model(_if, "if_model")
    mlflow.log_dict({"pca_components": _npc, "feature_cols": _feat}, "cluster_config.json")
    _track_pipeline("P5_PCA_IF_KMeans", _sil, run.info.run_id)
_profile(_d_clean, "cluster", _feat, f"{_DEV} PCA+IF+KM")
print(f"\n  [CELL 18 : [{_DEV}] Pipeline 5 PCA+IF+KMeans]  Elapsed: {(_time.time()-_t0)/60:.2f} min")


In [ ]:
# ── CELL 12 : [TVM] Pipeline 6 — DBSCAN ──────────────────────────────────
import time as _time; _t0 = _time.time()
_DEV = "TVM"
print("\n" + "="*70)
print(f"  {_DEV}  |  Pipeline 6: DBSCAN (Density-Based Spatial Clustering)")
print("  Step 1: Scale features; find eps via k-distance plot (KneeLocator).")
print("  Step 2: DBSCAN auto-determines cluster count from local density.")
print("  Noise  : Devices not in any cluster labelled cluster = -1.")
print("  Purpose: Non-spherical clusters; flags isolated / rare-fault devices.")
print("="*70)

_d = _samp(_DEV, S_DBSCAN)
_Xs, _feat = _scale(_d)
print(f"\n  Rows: {len(_d):,}   |   Features: {len(_feat)}")

# k-distance to estimate eps
_nbrs  = NearestNeighbors(n_neighbors=DBSCAN_K_NEIGHBORS).fit(_Xs)
_kdist = np.sort(_nbrs.kneighbors(_Xs)[0][:, -1])[::-1]
_kl_db = KneeLocator(range(len(_kdist)), _kdist, curve="convex", direction="decreasing")
_eps   = float(_kdist[_kl_db.knee]) if _kl_db.knee else float(np.percentile(_kdist, 10))
print(f"  Auto eps (knee of k-dist): {_eps:.4f}")

_d = _d.copy()
_d["cluster"] = DBSCAN(eps=_eps, min_samples=DBSCAN_MIN_SAMPLES, n_jobs=-1).fit_predict(_Xs)
_n_cl    = len(set(_d["cluster"].unique()) - {-1})
_n_noise = (_d["cluster"] == -1).sum()
print(f"  Clusters found: {_n_cl}   |   Noise points: {_n_noise:,}  ({_n_noise/len(_d):.1%})")
_sil = (silhouette_score(_Xs[_d["cluster"].values != -1],
                         _d.loc[_d["cluster"] != -1, "cluster"])
        if _n_cl > 1 else float("nan"))
print(f"  Silhouette (excl. noise): {_sil:.4f}" if _sil == _sil else "  Silhouette: N/A")

_sum = _cluster_summary(_d, "cluster", _feat)
print(f"\n{'-'*60}\n  CLUSTER SUMMARY - {_DEV} DBSCAN\n{'-'*60}")
print(_sum.to_string(index=False))
_print_device_list(_d, "cluster", f"{_DEV} DBSCAN")

_X2     = _pca2(_Xs)
_nc_pl  = len(_d["cluster"].unique())
_pal    = sns.color_palette("tab10", max(_nc_pl, 1))
_all_cl = sorted(_d["cluster"].unique())

_fig = plt.figure(figsize=(22, 15))
_fig.suptitle(
    f"{_DEV} - Pipeline 6: DBSCAN  |  eps={_eps:.3f}  "
    f"min_samples={DBSCAN_MIN_SAMPLES}  clusters={_n_cl}  noise={_n_noise:,}",
    fontsize=14, fontweight="bold")
_gs = gridspec.GridSpec(3, 3, figure=_fig, hspace=0.50, wspace=0.38)

_ax0 = _fig.add_subplot(_gs[0, 0])
_ax0.plot(_kdist, lw=1.2, color="#1565C0")
if _kl_db.knee:
    _ax0.axvline(_kl_db.knee, color="red", ls="--", lw=1.5,
                 label=f"knee eps={_eps:.3f}")
_ax0.set(title="k-Distance Plot", xlabel="Points (sorted desc)",
         ylabel=f"{DBSCAN_K_NEIGHBORS}-NN dist")
_ax0.legend(fontsize=8)

_ax1 = _fig.add_subplot(_gs[0, 1])
for i, cl in enumerate(_all_cl):
    m   = _d["cluster"].values == cl
    col = "#999999" if cl == -1 else _pal[i % 10]
    lbl = f"Noise({m.sum()})" if cl == -1 else f"C{cl}({m.sum()})"
    _ax1.scatter(_X2[m, 0], _X2[m, 1], c=[col], alpha=0.35, s=6, label=lbl)
_ax1.legend(markerscale=3, fontsize=7, ncol=2)
_ax1.set(title="PCA-2D DBSCAN Clusters", xlabel="PC1", ylabel="PC2")

_ax_bar(_fig.add_subplot(_gs[0, 2]), _d["cluster"].values, _nc_pl, "Records per Cluster")
_ax_anom(_fig.add_subplot(_gs[1, 0]), _d, "cluster", _nc_pl, "Failure Rate per Cluster")
_ax_heatmap(_fig.add_subplot(_gs[1, 1]), _d, "cluster", _feat, "Feature Means Heatmap")
_ax_devcount(_fig.add_subplot(_gs[1, 2]), _d, "cluster", "Unique Devices per Cluster")
_ax_timeline(_fig.add_subplot(_gs[2, 0]), _d, "cluster", "Monthly Records by Cluster")
_ax_summary_text(_fig.add_subplot(_gs[2, 1:3]), _sum, f"{_DEV} DBSCAN Summary")
_fig.tight_layout()
_save(_fig, f"{_DEV.lower()}_6_dbscan.png")
with mlflow.start_run(run_name=f"{_DEV}_P6_DBSCAN", tags={**COMMON_TAGS, "device_type": _DEV, "pipeline": "DBSCAN"}):
    mlflow.log_params({"algorithm": "DBSCAN", "eps": round(_eps, 4), "min_samples": DBSCAN_MIN_SAMPLES, "k_neighbors": DBSCAN_K_NEIGHBORS})
    mlflow.log_metrics({"silhouette_score": round(float(_sil) if _sil==_sil else 0.0, 4), "n_clusters": _n_cl, "n_noise": int(_n_noise), "noise_pct": round(100*_n_noise/len(_d), 2), "n_devices": len(_d)})
    _cdist = {f"cluster_{k}_size": int((_d['cluster']==k).sum()) for k in sorted(_d['cluster'].unique())}
    mlflow.log_metrics({f"cluster_{k}_size": v for k, v in _cdist.items()})
    mlflow.log_artifact(f"{OUT_DIR}/{_DEV.lower()}_6_dbscan.png")
_profile(_d, "cluster", _feat, f"{_DEV} DBSCAN")
print(f"\n  [Pipeline 6 DBSCAN - {_DEV}]  Elapsed: {(_time.time()-_t0)/60:.2f} min")


In [ ]:
# ── CELL 13 : [TVM] Pipeline 7 — Gaussian Mixture Model ─────────────────
import time as _time; _t0 = _time.time()
_DEV = "TVM"
print("\n" + "="*70)
print(f"  {_DEV}  |  Pipeline 7: Gaussian Mixture Model (GMM)")
print("  Algorithm : Expectation-Maximisation with full covariance matrices.")
print("  Optimal K : Minimum BIC across K range.")
print("  Benefit   : Soft assignments; handles elliptical / overlapping clusters.")
print("="*70)

_d = _samp(_DEV, S_GMM)
_Xs, _feat = _scale(_d)
print(f"\n  Rows: {len(_d):,}   |   Features: {len(_feat)}")

_bics = []
_kr   = range(K_MIN, K_MAX + 1)
print("  BIC search:", end=" ", flush=True)
for _k in _kr:
    _b = GaussianMixture(n_components=_k, covariance_type="full",
                         random_state=RANDOM_STATE, max_iter=200).fit(_Xs).bic(_Xs)
    _bics.append(_b)
    print(f"K={_k}({_b:.0f})", end=" ", flush=True)
print()
_opt = int(list(_kr)[int(np.argmin(_bics))])
print(f"  Optimal K (min BIC): {_opt}")

_gmm = GaussianMixture(n_components=_opt, covariance_type="full",
                       random_state=RANDOM_STATE, max_iter=200)
_d = _d.copy()
_d["cluster"]        = _gmm.fit_predict(_Xs)
_d["gmm_confidence"] = _gmm.predict_proba(_Xs).max(axis=1)
_sil = silhouette_score(_Xs, _d["cluster"]) if _opt > 1 else float("nan")
print(f"  Silhouette: {_sil:.4f}   |   Mean confidence: {_d['gmm_confidence'].mean():.3f}")

_sum = _cluster_summary(_d, "cluster", _feat)
print(f"\n{'-'*60}\n  CLUSTER SUMMARY - {_DEV} GMM\n{'-'*60}")
print(_sum.to_string(index=False))
_print_device_list(_d, "cluster", f"{_DEV} GMM")

_X2 = _pca2(_Xs)

_fig = plt.figure(figsize=(22, 15))
_fig.suptitle(
    f"{_DEV} - Pipeline 7: GMM (full cov)  |  "
    f"K={_opt}  BIC={min(_bics):.0f}  sil={_sil:.3f}",
    fontsize=14, fontweight="bold")
_gs = gridspec.GridSpec(3, 3, figure=_fig, hspace=0.50, wspace=0.38)

_ax0 = _fig.add_subplot(_gs[0, 0])
_ax0.plot(list(_kr), _bics, "bo-", ms=5)
_ax0.axvline(_opt, color="red", ls="--", lw=1.5, label=f"K={_opt}")
_ax0.set(title="BIC vs K (lower = better)", xlabel="K", ylabel="BIC")
_ax0.legend()

_ax_cl(_fig.add_subplot(_gs[0, 1]), _X2, _d["cluster"].values, _opt,
       f"PCA-2D GMM Clusters  sil={_sil:.3f}" if _sil == _sil else "PCA-2D GMM Clusters",
       "PC1", "PC2")

_ax2 = _fig.add_subplot(_gs[0, 2])
_ax2.hist(_d["gmm_confidence"], bins=30, color="#1565C0", alpha=0.8)
_ax2.set(title="GMM Assignment Confidence", xlabel="Max Probability", ylabel="Count")

_ax_anom(_fig.add_subplot(_gs[1, 0]), _d, "cluster", _opt, "Failure Rate per Cluster")
_ax_heatmap(_fig.add_subplot(_gs[1, 1]), _d, "cluster", _feat, "Feature Means Heatmap")
_ax_devcount(_fig.add_subplot(_gs[1, 2]), _d, "cluster", "Unique Devices per Cluster")
_ax_bar(_fig.add_subplot(_gs[2, 0]), _d["cluster"].values, _opt, "Records per Cluster")
_ax_timeline(_fig.add_subplot(_gs[2, 1]), _d, "cluster", "Monthly Records by Cluster")
_ax_summary_text(_fig.add_subplot(_gs[2, 2]), _sum, f"{_DEV} GMM Summary")
_fig.tight_layout()
_save(_fig, f"{_DEV.lower()}_7_gmm.png")
with mlflow.start_run(run_name=f"{_DEV}_P7_GMM", tags={**COMMON_TAGS, "device_type": _DEV, "pipeline": "GMM"}) as run:
    mlflow.log_params({"algorithm": "GMM", "n_clusters": _opt, "covariance_type": "full", "max_iter": 200, "k_min": K_MIN, "k_max": K_MAX})
    mlflow.log_metrics({"silhouette_score": round(float(_sil) if _sil==_sil else 0.0, 4), "bic_optimal": round(min(_bics), 2), "mean_confidence": round(float(_d['gmm_confidence'].mean()), 4), "n_devices": len(_d)})
    _cdist = {f"cluster_{k}_size": int((_d['cluster']==k).sum()) for k in range(_opt)}
    mlflow.log_metrics(_cdist)
    mlflow.log_artifact(f"{OUT_DIR}/{_DEV.lower()}_7_gmm.png")
    _sk_scaler = StandardScaler().fit(_d[_feat].values)
    _sk_gmm = GaussianMixture(n_components=_opt, covariance_type="full", random_state=RANDOM_STATE, max_iter=200).fit(_sk_scaler.transform(_d[_feat].values))
    _sk_pipe = skPipeline([("scaler", _sk_scaler), ("gmm", _sk_gmm)])
    mlflow.sklearn.log_model(_sk_pipe, "model")
    _track_pipeline("P7_GMM", _sil, run.info.run_id)
_profile(_d, "cluster", _feat, f"{_DEV} GMM")
print(f"\n  [Pipeline 7 GMM - {_DEV}]  Elapsed: {(_time.time()-_t0)/60:.2f} min")


In [ ]:
# ── CELL 14 : MLflow Model Registry — TVM champion (best silhouette) ────────
import boto3
from botocore.exceptions import ClientError
from mlflow.tracking import MlflowClient

_mlflow_client = MlflowClient()
_sm_client = boto3.client("sagemaker", region_name=REGION)
_today = pd.Timestamp.today().strftime("%Y-%m-%d")

_candidates = [(k, v) for k, v in PIPELINE_RESULTS.items()
               if v.get("run_id") and v.get("sil") is not None and v["sil"] == v["sil"]]
if not _candidates:
    _exp = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT)
    if _exp:
        _runs = _mlflow_client.search_runs(
            experiment_ids=[_exp.experiment_id],
            filter_string=f"tags.device_type = 'TVM' and attributes.start_time >= '{_today}'",
            order_by=["metrics.silhouette_score DESC"],
            max_results=10,
        )
        if _runs:
            _best = _runs[0]
            _champ_name = _best.data.tags.get("pipeline", "unknown")
            _champ_run_id = _best.info.run_id
            _champ_sil = float(_best.data.metrics.get("silhouette_score", 0))
        else:
            _champ_name = _champ_run_id = _champ_sil = None
    else:
        _champ_name = _champ_run_id = _champ_sil = None
else:
    _champ_name, _champ = max(_candidates, key=lambda x: x[1]["sil"])
    _champ_run_id = _champ["run_id"]
    _champ_sil = _champ["sil"]

print(f"\n{'='*70}\n  TVM Model Registry\n{'='*70}")
if not _champ_run_id:
    print("  [WARN] No champion run with logged model — run P1/P2/P5/P7 first")
else:
    print(f"  Champion pipeline : {_champ_name}")
    print(f"  Silhouette        : {_champ_sil:.4f}")
    print(f"  Run ID            : {_champ_run_id}")
    _model_uri = f"runs:/{_champ_run_id}/model"
    try:
        _mv = mlflow.register_model(model_uri=_model_uri, name=MLFLOW_MODEL_NAME)
        print(f"  MLflow registered : {MLFLOW_MODEL_NAME} v{_mv.version}")
        for _tk, _tv in {
            "device_type": "TVM", "pipeline": str(_champ_name), "flavor": "sklearn",
            "silhouette_score": f"{_champ_sil:.4f}", "run_date": _today,
        }.items():
            _mlflow_client.set_model_version_tag(MLFLOW_MODEL_NAME, _mv.version, _tk, _tv)
        for _ev in _mlflow_client.search_model_versions(f"name='{MLFLOW_MODEL_NAME}'"):
            if _ev.current_stage == "Staging" and _ev.version != _mv.version:
                _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _ev.version, "Archived")
        _mlflow_client.transition_model_version_stage(MLFLOW_MODEL_NAME, _mv.version, "Staging")
        print(f"  Promoted          : {MLFLOW_MODEL_NAME} v{_mv.version} -> Staging")
    except Exception as _e:
        print(f"  [WARN] MLflow register_model failed: {_e}")
        _mv = None

    try:
        _sm_client.describe_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME)
    except ClientError:
        _sm_client.create_model_package_group(
            ModelPackageGroupName=MODEL_REGISTRY_NAME,
            ModelPackageGroupDescription=f"PS4 TVM fault clustering — {MLFLOW_MODEL_NAME}",
        )
        print(f"  Created SM group  : {MODEL_REGISTRY_NAME}")

    if _mv is not None:
        try:
            _pkg = _sm_client.create_model_package(
                ModelPackageGroupName=MODEL_REGISTRY_NAME,
                ModelPackageDescription=f"TVM champion {_champ_name} sil={_champ_sil:.4f}",
                ModelApprovalStatus="PendingManualApproval",
                MetadataProperties={"GeneratedBy": "PS4_FaultClustering_TVM"},
                CustomerMetadataProperties={
                    "device_type": "TVM", "pipeline": str(_champ_name),
                    "silhouette_score": f"{_champ_sil:.4f}", "mlflow_version": str(_mv.version),
                },
                ModelMetrics=[{
                    "Name": "silhouette_score", "Value": float(_champ_sil),
                    "Type": "number", "Unit": "none",
                }],
            )
            print(f"  SM Model Package  : {_pkg['ModelPackageArn']}")
        except Exception as _e:
            print(f"  [WARN] SageMaker Model Package failed: {_e}")



In [ ]:
# ── CELL 16 : Export fault-clustering results to S3 (optional) ───────────────
# Run after CELL 14 (champion selected). Skips safely if ENABLE_S3_EXPORT=False.
# Writes: s3://{bucket}/chicago/ps4/clustering/<gate|tvm|validator>/asof=<date>/
#         assignments/ + cluster_summary/ + manifest JSON for RDS loader.

import sys
from pathlib import Path

if not ENABLE_S3_EXPORT:
    print("[SKIP] ENABLE_S3_EXPORT=False")
elif not globals().get("_champ_run_id"):
    print("[SKIP] Run CELL 14 first — champion run_id not set")
else:
    _helper_paths = [
        Path.cwd(),
        Path.cwd() / "ps4_fault_clustering",
        Path("/home/sagemaker-user/PS4_Clustering"),
        Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd(),
    ]
    for _hp in _helper_paths:
        if (_hp / "ps4_cluster_s3_export.py").is_file():
            sys.path.insert(0, str(_hp))
            break
    from ps4_cluster_s3_export import export_fault_clusters_to_s3
    import mlflow.sklearn

    print(f"[S3] Loading champion model  pipeline={_champ_name}  run={_champ_run_id[:8]}...")
    _champ_model = mlflow.sklearn.load_model(f"runs:/{_champ_run_id}/model")
    _export_df = _samp(_DEV, None)
    _manifest = export_fault_clusters_to_s3(
        df_devices=_export_df,
        feature_cols=FEATURE_COLS,
        model=_champ_model,
        device_type=_DEV,
        champion_pipeline=str(_champ_name),
        champion_run_id=str(_champ_run_id),
        champion_silhouette=float(_champ_sil),
        bucket=S3_BUCKET,
        prefix=PS4_CLUSTER_S3_PREFIX,
        city_code=CITY_CODE,
        region=REGION,
        device_id_col=DEVICE_ID_COL,
        device_key_col="DEVICE_KEY",
        target_col=TARGET_COL,
    )
    print(f"[S3] Exported {_manifest['n_devices']:,} devices  "
          f"{_manifest['n_clusters']} clusters  asof={_manifest['asof_date']}")
    print(f"  assignments -> {_manifest['paths']['assignments']}")
    print(f"  summary     -> {_manifest['paths']['cluster_summary']}")
    print(f"  manifest    -> {_manifest['manifest_uri']}")


In [ ]:
import subprocess, os, datetime
_nb_path  = "/home/sagemaker-user/PS4_Clustering/PS4_FaultClustering_TVM.ipynb"
_ts       = datetime.datetime.now().strftime("%Y%m%d_%H%M")
_out_html = f"/home/sagemaker-user/PS4_Clustering/PS4_FaultClustering_TVM_output_{_ts}.html"
if os.path.exists(_nb_path):
    _result = subprocess.run(
        ["jupyter", "nbconvert", "--to", "html", _nb_path, "--output", _out_html],
        capture_output=True, text=True)
    if _result.returncode == 0:
        _size = os.path.getsize(_out_html) / 1024
        print(f"[HTML EXPORT] Saved -> {_out_html}  ({_size:.0f} KB)")
    else:
        print(_result.stderr[-500:] if _result.stderr else "(no stderr)")
else:
    print(f"[HTML EXPORT] Notebook not found at {_nb_path}")
    print(f"  Run: !find /home/sagemaker-user -name 'PS4_FaultClustering_TVM.ipynb' to locate it.")
